In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
### Padding to the next power of 2
def next_power_of_two(m: int) -> int:
    if m <= 0:
        raise ValueError("m must be positive")
    return 1 << (m - 1).bit_length()
h = 1
def fwht_inplace(A: np.ndarray) -> np.ndarray:
    """
    In-place UNNORMALIZED Walsh–Hadamard transform along axis 0.
    Requires number of rows to be a power of 2.
    """
    n = A.shape[0]
    if n & (n - 1) != 0:
        raise ValueError(f"FWHT length must be a power of 2, got {n}")

    h = 1
    while h < n:
        for i in range(0, n, 2 * h):
            a = A[i:i+h, ...].copy()
            b = A[i+h:i+2*h, ...].copy()
            A[i:i+h, ...] = a + b
            A[i+h:i+2*h, ...] = a - b
        h *= 2
    return A


def rht_local(X: np.ndarray, Y: np.ndarray, rng: np.random.Generator):
    """
    Local RHT:  (1/sqrt(m)) * H_m * D_m applied to rows of X and Y.
    Assumes X,Y already padded to m (power of 2).
    """
    m = X.shape[0]
    signs = rng.choice(np.array([-1.0, 1.0]), size=m, replace=True)
    Xd = X * signs[:, None]
    Yd = Y * signs[:, None]

    fwht_inplace(Xd)
    fwht_inplace(Yd)

    scale = 1.0 / np.sqrt(m)
    Xd *= scale
    Yd *= scale
    return Xd, Yd

In [3]:
def baseline_uniform_local_rate(X_raw_list, Y_raw_list, s, rng):
    X_parts, Y_parts = [], []
    for Xi, Yi in zip(X_raw_list, Y_raw_list):
        mi = Xi.shape[0]
        ri = int(np.round(mi / s))
        ri = max(1, ri)  # optional: avoid empty transmission
        replace = (mi < ri)
        idx = rng.choice(mi, size=ri, replace=replace)
        X_parts.append(Xi[idx])
        Y_parts.append(Yi[idx])
    return np.vstack(X_parts), np.vstack(Y_parts)

In [4]:
# ---------- Distribution plugin (default: GMM with SPD covariances) ----------
def random_spd_covariance(
    p: int,
    jitter: float = 1e-3
 ):
    """Generate a random symmetric positive definite covariance matrix."""
    A = np.random.normal(size=(p, p))
    Sigma = (A @ A.T) / p
    Sigma += jitter * np.eye(p)
    return Sigma

def sample_gmm_client_X_only(mi: int, p: int,
                              weight: float,
                              mu_1: float, mu_2: float,
                              Sigma_1: np.ndarray, covariance_scale_2: float):
    """
    Sample X_i on one client from a 2-component GMM (design matrix only, no Y).
      X ~ weight * N(mu_1 1, Sigma_1) + (1-weight) * N(mu_2 1, Sigma_2)
      where Sigma_2 = covariance_scale_2 * Sigma_1
    Returns:
      X: (mi, p)
    """
    if Sigma_1.shape != (p, p):
        raise ValueError(f"Sigma_1 must have shape {(p, p)}, got {Sigma_1.shape}")
    if covariance_scale_2 <= 0:
        raise ValueError("covariance_scale_2 must be positive")

    Sigma_2 = covariance_scale_2 * Sigma_1
    mean_1 = np.full(p, mu_1, dtype=float)
    mean_2 = np.full(p, mu_2, dtype=float)

    z = np.random.random(mi) < weight
    n1 = int(np.sum(z))
    n2 = mi - n1

    X = np.empty((mi, p), dtype=float)
    if n1 > 0:
        X[z] = np.random.multivariate_normal(mean=mean_1, cov=Sigma_1, size=n1)
    if n2 > 0:
        X[~z] = np.random.multivariate_normal(mean=mean_2, cov=Sigma_2, size=n2)

    return X

In [5]:
import numpy as np

def multinomial_no_zeros(n: int, k: int) -> np.ndarray:
    """Guarantee m_i >= 1 by allocating 1 sample per client, then multinomial for remaining.
    Uses softmax of Gaussian draws to create heterogeneous client probabilities."""
    rng = np.random.default_rng()
    if n < k:
        raise ValueError(f"Need n >= k to guarantee non-empty clients, got n={n}, k={k}")
    # Generate heterogeneous probabilities via softmax of Gaussian noise
    logits = rng.normal(loc=0.0, scale=1.0, size=k)
    probs = np.exp(logits - logits.max())
    probs /= probs.sum()
    z = rng.multinomial(n - k, probs)
    return z + 1

def step1_generate_clients_X_only(
    n: int,
    k: int,
    p: int,
    weight: float, mu_1: float, mu_2: float,
    covariance_scale_2: float,
    sampler_fn
):
    """
    Step 1 (design matrix only — no Y):
      - Sample heterogeneous client sizes m_i.
      - Generate one random SPD covariance Sigma_1.
      - Set Sigma_2 = covariance_scale_2 * Sigma_1.
      - Generate local X_i with m_i rows (NO Y — Y is generated fresh each rep).
      - Compute m_loc = next_power_of_two(max_i m_i).
      - Pad each client X to m_loc rows with zeros (for FWHT).

    Returns:
      X_pad_list : lists length k, each (m_loc, p)
      X_raw_list : lists length k, each (m_i, p)
      m_sizes    : (k,) int
      m_max, m_loc : int
      beta_true  : (p,)
      Sigma_1, Sigma_2 : (p, p) covariance matrices
    """
    if n <= 0 or k <= 0 or p <= 0:
        raise ValueError("n, k, p must be positive integers")

    beta_true = np.random.normal(0.0, 1.0, size=p)
    Sigma_1 = random_spd_covariance(p)
    Sigma_2 = covariance_scale_2 * Sigma_1
    m_sizes = multinomial_no_zeros(n, k)

    m_max = int(m_sizes.max())
    m_loc = next_power_of_two(m_max)

    X_pad_list = []
    X_raw_list = []

    for i in range(k):
        mi = int(m_sizes[i])

        Xi = sampler_fn(
            mi, p,
            weight, mu_1, mu_2,
            Sigma_1, covariance_scale_2
        )  # Xi: (mi, p)

        X_raw_list.append(Xi)

        if mi < m_loc:
            pad = m_loc - mi
            Xi_pad = np.vstack([Xi, np.zeros((pad, p), dtype=float)])
        else:
            Xi_pad = Xi

        X_pad_list.append(Xi_pad)

    return X_pad_list, X_raw_list, m_sizes, m_max, m_loc, beta_true, Sigma_1, Sigma_2

In [6]:
import numpy as np

def local_shuffle_rht_mask_send(
    X_list, Y_list,
    m_loc: int,
    proportion: float,
    random_seed: int = 39
):
    """
    DGSRHT Algorithm 1: Steps 2–6, implemented in a communication-faithful way.

    Paper Steps:
      Step 2: local shuffle
      Step 3: local RHT  (H_{m_loc} D_{m_loc})
      Step 4: server draws ONE shared subsampling mask B with keep rate 1/s
      Step 5: each client applies B
      Step 6: each client sends \tilde{X}'_i, \tilde{Y}'_i to server

    Implementation detail:
      Instead of sending an (m_loc, p) matrix with many zero rows (B * \tilde{X}_i),
      we send only the kept rows indexed by keep_idx (equivalent after zero-row removal).

    Returns
    -------
    X_send_list, Y_send_list : lists length K
        Each X_send_list[i] is (r, p) and Y_send_list[i] is (r, 1), where r = m_loc/s.
        These are the nonzero rows of B * \tilde{X}_i and B * \tilde{Y}_i.
    keep_idx : np.ndarray shape (r,)
        Shared kept row indices (defines B).
    s : int
        s = 1/proportion (power-of-two).
    """
    K = len(X_list)
    if len(Y_list) != K:
        raise ValueError("X_list and Y_list must have the same length (K clients).")

    # ---- Step 4: shared mask B with keep fraction 1/s ----
    s = int(round(1.0 / proportion))
    if not np.isclose(proportion, 1.0 / s):
        raise ValueError(f"proportion must be exactly 1/s. Got proportion={proportion}, but 1/proportion≈{1.0/proportion}.")
    if s <= 0 or (s & (s - 1)) != 0:
        raise ValueError(f"s must be a positive power-of-two. Got s={s}.")
    if s > m_loc:
        raise ValueError(f"s={s} must satisfy s <= m_loc={m_loc}.")

    r = m_loc // s
    rng_mask = np.random.default_rng(random_seed)
    keep_idx = rng_mask.choice(m_loc, size=r, replace=False)

    # ---- Steps 2–3–5–6: shuffle, RHT, subsample-and-send ----
    X_send_list, Y_send_list = [], []

    for i in range(K):
        Xi = X_list[i]
        Yi = Y_list[i]

        if Xi.shape[0] != m_loc or Yi.shape[0] != m_loc:
            raise ValueError(f"Client {i}: expected m_loc rows={m_loc}, got X rows={Xi.shape[0]}, Y rows={Yi.shape[0]}.")
        if Yi.ndim != 2 or Yi.shape[1] != 1:
            raise ValueError(f"Client {i}: Y must be shaped (m_loc, 1). Got {Yi.shape}.")

        # Step 2: local shuffle
        rng_perm = np.random.default_rng(random_seed + 20 + i)
        perm = rng_perm.permutation(m_loc)
        Xi_shuf = Xi[perm].copy()
        Yi_shuf = Yi[perm].copy()

        # Step 3: local RHT (uses your helper: (1/sqrt(m)) H_m D_m )
        rng_sign = np.random.default_rng(random_seed + 30 + i)
        X_tilde, Y_tilde = rht_local(Xi_shuf, Yi_shuf, rng_sign)

        # Step 5 + Step 6: apply shared mask B and send only kept rows
        X_send = X_tilde[keep_idx, :]
        Y_send = Y_tilde[keep_idx, :]

        X_send_list.append(X_send)
        Y_send_list.append(Y_send)

    return X_send_list, Y_send_list, keep_idx, s, r

In [7]:
def server_mixer(X_send_list, Y_send_list,s,K,r):
    # step 6: Stack
    X_tilde_prime = np.vstack(X_send_list)
    Y_tilde_prime = np.vstack(Y_send_list)
    # Step 7: DGSRHT Schema
    p = X_tilde_prime.shape[1]
    
    # (K*r, p) -> (K, r, p) -> (K, r*p)
    X_flat = X_tilde_prime.reshape(K, r, p).reshape(K, r * p).copy()
    Y_flat = Y_tilde_prime.reshape(K, r, 1).reshape(K, r).copy()  # (K, r)

    # Unnormalized Hadamard across clients
    fwht_inplace(X_flat)
    fwht_inplace(Y_flat)

    # Normalize to orthonormal H_K
    scale = 1.0 / np.sqrt(K)
    X_flat *= scale
    Y_flat *= scale

    # Back to stacked layout (K*r, p) and (K*r, 1)
    X_star = X_flat.reshape(K, r, p).reshape(K * r, p)
    Y_star = Y_flat.reshape(K, r, 1).reshape(K * r, 1)
    star_scale = np.sqrt(s)
    X_star *= star_scale
    Y_star *= star_scale
    return X_star, Y_star

In [8]:
def partition_matrix_not_necessary_divisable(X,Y,k):
    """
    Partition the rows of matrix X across k machines uniformly with equal probability.
    Each machine will receive a sub-matrix of X and the corresponding sub-vector of Y.

    Parameters:
    X (numpy.ndarray): The input matrix to be partitioned (n rows, m columns).
    Y (numpy.ndarray): The output vector to be partitioned (n rows, 1 column).
    k (int): The number of machines.

    Returns:
    list of numpy.ndarray: A list of sub-matrices, each assigned to a machine.
    """
    # Get the number of rows in the matrix
    n = X.shape[0]
    
    # Create an array of row indices (0 to n-1)
    indices = np.arange(n)
    
    # Shuffle the indices randomly to ensure each row has equal probability
    np.random.shuffle(indices)
    
    # Calculate the base size of each partition and the remainder
    part_size = n // k  # Base number of rows per machine
    remainder = n % k   # Extra rows to distribute
    
    # Split the indices into k parts
    parts = []
    start = 0
    for i in range(k):
        # The first 'remainder' machines get one extra row
        end = start + part_size + (1 if i < remainder else 0)
        parts.append(indices[start:end])
        start = end
    
    # Create sub-matrices for each machine using the partitioned indices
    sub_matrices = [X[part] for part in parts]
    sub_matrices_Y = [Y[part] for part in parts]
    
    return sub_matrices, sub_matrices_Y

In [9]:
def average_regression_train_not_divisable(X, Y, k):
    """
    Distributed OLS with uniform random partition into k machines (not necessarily divisible).
    Uses your partition_matrix_not_necessary_divisable.
    Returns the averaged beta across machines.
    """
    sub_matrices, sub_target = partition_matrix_not_necessary_divisable(X, Y, k)
    beta_list = []

    for X_i, Y_i in zip(sub_matrices, sub_target):
        X_i = np.asarray(X_i)
        Y_i = np.asarray(Y_i).reshape(-1)
        # local OLS (no regularization)
        beta_i = np.linalg.inv(X_i.T @ X_i) @ (X_i.T @ Y_i)
        beta_list.append(beta_i)

    beta_avg = np.mean(beta_list, axis=0)
    return beta_avg

In [10]:
def MSE(X, A, delta=1.0):
    """
    MSE(X, A, delta) = delta^2 * tr( (X^T X)^{-1} A^T A ).
    For A = I_p, this is delta^2 * tr( (X^T X)^{-1} ).
    """
    XtX = X.T @ X
    XtX_inv = np.linalg.inv(XtX)
    return (delta**2) * np.trace(XtX_inv @ (A.T @ A))


def average_MSE_expectation_theory(X, Y, k, delta=1.0):
    """
    Empirical version of E[MSE] for the averaged estimator over k machines,
    using ONE random partition of (X, Y) into k blocks.

    MSE for each block i:  MSE(X_i, I_p, delta)
    Averaged estimator: beta_bar = (1/k) sum_i beta_i
    => variance scales with (1/k)^2, so multiply each block MSE by (1/k)^2.
    """
    sub_matrices, sub_target = partition_matrix_not_necessary_divisable(X, Y, k)
    mse_pre = 0.0
    for X_i in sub_matrices:
        X_i = np.asarray(X_i)
        mse_i = MSE(X_i, np.eye(X_i.shape[1]), delta)
        mse_pre += (1.0 / k)**2 * mse_i
    return mse_pre


def Relative_Efficiency_theory(MSE_ols, X, Y, k, delta=1.0):
    """
    RE_theory(k) = MSE_ols / E[MSE_avg(k)]
    where MSE_ols is the centralized OLS MSE on the same design X.
    """
    mse = average_MSE_expectation_theory(X, Y, k, delta)
    return MSE_ols / mse


In [11]:
def run_experiment_DGSRHT(
    n, k, p,
    weight, mu_1, mu_2,
    covariance_scale_2,
    proportion,
    K_prime_list,
    R=100,
    base_seed=10
):
    """
    OLS simulation: X is fixed, epsilon (hence Y) is resampled each repetition.

    Setup (done once):
      - Generate heterogeneous client X data via step1_generate_clients_X_only.
      - This fixes X_raw_list, X_pad_list, beta_true, m_sizes, m_loc.

    For each repetition r = 1..R:
      - Draw fresh epsilon_i for each client -> Y_raw_i = X_raw_i @ beta + eps_i.
      - Pad Y to m_loc.
      - Run DGSRHT pipeline  -> X_star, Y_star.
      - Run uniform baseline -> X_uniform, Y_uniform.
      - Centralized & distributed OLS -> MSE vs beta_true.

    Theoretical RE is computed ONCE after the loop.

    Returns four dicts K' -> RE(K'):
      RE_uniform_emp, RE_DGSRHT_emp, RE_DGSRHT_theory, RE_uniform_theory
    """
    K_prime_list = list(K_prime_list)
    s_val = int(round(1.0 / proportion))

    # ---- Generate X once (fixed design) ----
    (X_pad_list, X_raw_list,
     m_sizes, m_max, m_loc,
     beta_true, Sigma_1, Sigma_2
    ) = step1_generate_clients_X_only(
        n=n, k=k, p=p,
        weight=weight, mu_1=mu_1, mu_2=mu_2,
        covariance_scale_2=covariance_scale_2,
        sampler_fn=sample_gmm_client_X_only
    )

    # Accumulators
    MSE_ols_uniform_sum = 0.0
    MSE_ols_DGSRHT_sum  = 0.0
    MSE_uniform_sum     = {Kp: 0.0 for Kp in K_prime_list}
    MSE_DGSRHT_sum      = {Kp: 0.0 for Kp in K_prime_list}

    X_star_first    = None
    Y_star_first    = None
    X_uniform_first = None
    Y_uniform_first = None

    for rep in range(R):
        seed_r = base_seed + rep
        rng_eps = np.random.default_rng(seed_r)

        # ---- Draw fresh epsilon -> fresh Y for each client ----
        Y_raw_list = []
        Y_pad_list = []
        for i in range(k):
            Xi_raw = X_raw_list[i]          # (m_i, p) — fixed
            mi = Xi_raw.shape[0]
            eps_i = rng_eps.normal(0.0, 1.0, size=mi)
            Yi_raw = (Xi_raw @ beta_true + eps_i).reshape(-1, 1)  # (m_i, 1)
            Y_raw_list.append(Yi_raw)

            if mi < m_loc:
                Yi_pad = np.vstack([Yi_raw, np.zeros((m_loc - mi, 1), dtype=float)])
            else:
                Yi_pad = Yi_raw
            Y_pad_list.append(Yi_pad)

        # ---- Steps 2-6: local shuffle + RHT + mask + send ----
        X_send_list, Y_send_list, keep_idx, s, r_keep = (
            local_shuffle_rht_mask_send(
                X_pad_list, Y_pad_list,
                m_loc=m_loc,
                proportion=proportion,
                random_seed=seed_r
            )
        )

        # ---- Step 7: server Hadamard mixing ----
        X_star, Y_star = server_mixer(
            X_send_list, Y_send_list, s=s, K=k, r=r_keep
        )

        # ---- Uniform baseline (same subsampling rate) ----
        rng_unif = np.random.default_rng(seed_r)
        X_uniform, Y_uniform = baseline_uniform_local_rate(
            X_raw_list, Y_raw_list, s=s_val, rng=rng_unif
        )

        # Save first designs for theory
        if X_star_first is None:
            X_star_first    = X_star.copy()
            Y_star_first    = Y_star.copy()
            X_uniform_first = X_uniform.copy()
            Y_uniform_first = Y_uniform.copy()

        # ---- Centralized OLS on uniform baseline ----
        Y_uniform_vec = Y_uniform.reshape(-1)
        beta_ols_u = np.linalg.inv(X_uniform.T @ X_uniform) @ (X_uniform.T @ Y_uniform_vec)
        MSE_ols_uniform_sum += np.sum((beta_ols_u - beta_true)**2)

        # ---- Centralized OLS on DGSRHT output ----
        Y_star_vec = Y_star.reshape(-1)
        beta_ols_d = np.linalg.inv(X_star.T @ X_star) @ (X_star.T @ Y_star_vec)
        MSE_ols_DGSRHT_sum += np.sum((beta_ols_d - beta_true)**2)

        # ---- Distributed OLS for each K' ----
        for Kp in K_prime_list:
            beta_u = average_regression_train_not_divisable(X_uniform, Y_uniform_vec, Kp)
            MSE_uniform_sum[Kp] += np.sum((beta_u - beta_true)**2)

            beta_d = average_regression_train_not_divisable(X_star, Y_star_vec, Kp)
            MSE_DGSRHT_sum[Kp] += np.sum((beta_d - beta_true)**2)

        if (rep + 1) % 25 == 0:
            print(f'  repetition {rep+1}/{R} done')

    # ---- Average over R repetitions ----
    MSE_ols_uniform_avg = MSE_ols_uniform_sum / R
    MSE_ols_DGSRHT_avg  = MSE_ols_DGSRHT_sum  / R

    RE_uniform_emp   = {}
    RE_DGSRHT_emp    = {}
    RE_DGSRHT_theory = {}
    RE_uniform_theory = {}

    for Kp in K_prime_list:
        mse_u = MSE_uniform_sum[Kp] / R
        mse_d = MSE_DGSRHT_sum[Kp]  / R
        RE_uniform_emp[Kp] = MSE_ols_uniform_avg / mse_u
        RE_DGSRHT_emp[Kp]  = MSE_ols_DGSRHT_avg  / mse_d

    # ---- Theoretical RE for DGSRHT ----
    MSE_ols_theory_DGSRHT = MSE(X_star_first, np.eye(p), delta=1.0)
    for Kp in K_prime_list:
        RE_DGSRHT_theory[Kp] = Relative_Efficiency_theory(
            MSE_ols=MSE_ols_theory_DGSRHT,
            X=X_star_first,
            Y=Y_star_first,
            k=Kp,
            delta=1.0
        )

    # ---- Theoretical RE for Uniform ----
    MSE_ols_theory_uniform = MSE(X_uniform_first, np.eye(p), delta=1.0)
    for Kp in K_prime_list:
        RE_uniform_theory[Kp] = Relative_Efficiency_theory(
            MSE_ols=MSE_ols_theory_uniform,
            X=X_uniform_first,
            Y=Y_uniform_first,
            k=Kp,
            delta=1.0
        )

    return RE_uniform_emp, RE_DGSRHT_emp, RE_DGSRHT_theory, RE_uniform_theory

In [ ]:
# === Experiment parameters ===
n = 8192 * 4          # total samples
k = 16                # number of clients in DGSRHT pipeline
p = 200               # feature dimension

weight = 0.8          # GMM mixture weight for component 1
mu_1, mu_2 = 0, 5     # means of the two Gaussian components
covariance_scale_2 = 10.0   # Sigma_2 = covariance_scale_2 * Sigma_1
proportion = 0.25     # keep fraction (1/s)

K_prime_list = np.arange(1, 31)  # number of working machines for distributed OLS
R = 100
base_seed = 10

RE_uniform_emp, RE_DGSRHT_emp, RE_DGSRHT_theory, RE_uniform_theory = run_experiment_DGSRHT(
    n=n, k=k, p=p,
    weight=weight,
    mu_1=mu_1, mu_2=mu_2,
    covariance_scale_2=covariance_scale_2,
    proportion=proportion,
    K_prime_list=K_prime_list,
    R=R,
    base_seed=base_seed
)

# Convert dicts -> arrays for plotting
Ks = np.array(K_prime_list)
RE_uniform_vals      = np.array([RE_uniform_emp[Kp]     for Kp in Ks])
RE_DGSRHT_emp_vals   = np.array([RE_DGSRHT_emp[Kp]      for Kp in Ks])
RE_DGSRHT_th_vals    = np.array([RE_DGSRHT_theory[Kp]   for Kp in Ks])
RE_uniform_th_vals   = np.array([RE_uniform_theory[Kp]  for Kp in Ks])

# === Plot: four lines on one graph ===
plt.figure(figsize=(10, 8))

plt.plot(Ks, RE_uniform_vals,    marker='o',          label=f'Uniform Sampling empirical (avg over {R})')
plt.plot(Ks, RE_uniform_th_vals, marker='d', ls='--', label='Uniform Sampling theoretical')
plt.plot(Ks, RE_DGSRHT_emp_vals, marker='s',          label=f'DGSRHT Algorithm empirical (avg over {R})')
plt.plot(Ks, RE_DGSRHT_th_vals,  marker='^', ls='--', label='DGSRHT Algorithm theoretical')

plt.xticks(Ks, Ks)
plt.xlabel(r'Number of working machines $K^\prime$')
plt.ylabel('Distributed Sketching Efficiency')
plt.title(
    f'Theoretical Sketching Relative Efficiency vs Empirical Sketching Efficiency\n'
    f'(n={n}, k={k}, p={p}, proportion={proportion}, '
    r'$\sigma^2_2$=' + f'{covariance_scale_2}, weight={weight}, R={R})'
)
plt.grid(axis='y', linestyle=':')
plt.legend()
plt.tight_layout()
plt.show()

  repetition 25/100 done


KeyboardInterrupt: 

In [ ]:
R = 300
base_seed = 15

RE_uniform_emp, RE_DGSRHT_emp, RE_DGSRHT_theory, RE_uniform_theory = run_experiment_DGSRHT(
    n=n, k=k, p=p,
    weight=weight,
    mu_1=mu_1, mu_2=mu_2,
    covariance_scale_2=covariance_scale_2,
    proportion=proportion,
    K_prime_list=K_prime_list,
    R=R,
    base_seed=base_seed
)

# Convert dicts -> arrays for plotting
Ks = np.array(K_prime_list)
RE_uniform_vals      = np.array([RE_uniform_emp[Kp]     for Kp in Ks])
RE_DGSRHT_emp_vals   = np.array([RE_DGSRHT_emp[Kp]      for Kp in Ks])
RE_DGSRHT_th_vals    = np.array([RE_DGSRHT_theory[Kp]   for Kp in Ks])
RE_uniform_th_vals   = np.array([RE_uniform_theory[Kp]  for Kp in Ks])

# === Plot: four lines on one graph ===
plt.figure(figsize=(10, 8))

plt.plot(Ks, RE_uniform_vals,    marker='o',          label=f'Uniform Sampling empirical (avg over {R})')
plt.plot(Ks, RE_uniform_th_vals, marker='d', ls='--', label='Uniform Sampling theoretical')
plt.plot(Ks, RE_DGSRHT_emp_vals, marker='s',          label=f'DGSRHT Algorithm empirical (avg over {R})')
plt.plot(Ks, RE_DGSRHT_th_vals,  marker='^', ls='--', label='DGSRHT Algorithm theoretical')

plt.xticks(Ks, Ks)
plt.xlabel(r'Number of working machines $K^\prime$')
plt.ylabel('Distributed Sketching Efficiency')
plt.title(
    f'Theoretical Sketching Relative Efficiency vs Empirical Sketching Efficiency\n'
    f'(n={n}, k={k}, p={p}, proportion={proportion}, '
    r'$\sigma^2_2$=' + f'{covariance_scale_2}, weight={weight}, R={R})'
)
plt.grid(axis='y', linestyle=':')
plt.legend()
plt.tight_layout()
plt.show()